# EN4554 — Assignment 2 on CNNs

**Name:** Lakshan R.G.R\
**Index number:** 220355M\
**GitHub:** https://github.com/ravindu374/EN4554_Assignment_2

Claude (Anthropic) was used as a coding assistant for this notebook. All code was reviewed, run
and verified by me, and I understand every line of it.

Discussion and interpretation of these results are in the accompanying report.

In [ ]:
import os, math, json, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision import datasets

# --- Windows + Python 3.10 SSL workaround -------------------------------------------
# Loading the Windows certificate store can fail with
#   ssl.SSLError: [ASN1: NOT_ENOUGH_DATA]
# if any certificate in it is malformed, which breaks the CIFAR-10 download.
# Passing an explicit CA file makes create_default_context skip the system store.
import ssl, certifi, functools
ssl._create_default_https_context = functools.partial(ssl.create_default_context,
                                                      cafile=certifi.where())

SEED = 0
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR    = 'data'
RESULTS_DIR = 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)

torch.backends.cudnn.benchmark = True   # autotune conv algorithms (inputs are a fixed size)

print('device:', DEVICE)

## Setup

Fixed seeds, a 45k/5k train/validation split, per-channel normalisation from the
training split only, and a JSON cache so results are not recomputed.

In [ ]:
FORCE_RETRAIN = False

def save_result(name, obj):
    with open(os.path.join(RESULTS_DIR, name + '.json'), 'w') as f:
        json.dump(obj, f, indent=1)

def load_result(name):
    path = os.path.join(RESULTS_DIR, name + '.json')
    if FORCE_RETRAIN or not os.path.exists(path):
        return None
    with open(path) as f:
        return json.load(f)

def cached(name, fn):
    """Return the cached result for `name`, otherwise run fn(), cache it and return it."""
    r = load_result(name)
    if r is None:
        t0 = time.time()
        r = fn()
        r['_seconds'] = round(time.time() - t0, 1)
        save_result(name, r)
    return r

In [ ]:
# ---- Load CIFAR-10 and build the 45k / 5k / 10k splits -----------------------------

train_full = datasets.CIFAR10(root=DATA_DIR, train=True,  download=True)
test_set   = datasets.CIFAR10(root=DATA_DIR, train=False, download=True)
CLASSES    = train_full.classes

# .data is a uint8 numpy array of shape (N, 32, 32, 3) -> we want (N, 3, 32, 32)
X_all = torch.from_numpy(train_full.data).permute(0, 3, 1, 2).contiguous()
Y_all = torch.tensor(train_full.targets)
X_te  = torch.from_numpy(test_set.data).permute(0, 3, 1, 2).contiguous()
Y_te  = torch.tensor(test_set.targets)

# One fixed permutation defines the split for the whole assignment
g    = torch.Generator().manual_seed(SEED)
perm = torch.randperm(len(X_all), generator=g)
tr_idx, va_idx = perm[:45000], perm[45000:]

X_tr, Y_tr = X_all[tr_idx], Y_all[tr_idx]
X_va, Y_va = X_all[va_idx], Y_all[va_idx]

# Per-channel mean/std from the TRAINING SPLIT ONLY, on the [0,1] scale
_tr_float = X_tr.float().div_(255.0)
MEAN = _tr_float.mean(dim=(0, 2, 3))
STD  = _tr_float.std(dim=(0, 2, 3))
del _tr_float

print('train %s   val %s   test %s' % (tuple(X_tr.shape), tuple(X_va.shape), tuple(X_te.shape)))
print('channel mean %s   std %s'
      % ([round(v, 4) for v in MEAN.tolist()], [round(v, 4) for v in STD.tolist()]))

# Move everything to the GPU as uint8 (138 MB) and keep the normalizer there too
X_tr, Y_tr = X_tr.to(DEVICE), Y_tr.to(DEVICE)
X_va, Y_va = X_va.to(DEVICE), Y_va.to(DEVICE)
X_te, Y_te = X_te.to(DEVICE), Y_te.to(DEVICE)   # NOT used until Task 4
MEAN_D = MEAN.view(1, 3, 1, 1).to(DEVICE)
STD_D  = STD.view(1, 3, 1, 1).to(DEVICE)

def normalize(xb_uint8):
    """uint8 (B,3,32,32) -> normalized float32 (B,3,32,32)."""
    return (xb_uint8.float().div_(255.0) - MEAN_D) / STD_D

def iterate(X, Y, batch_size, shuffle=True, generator=None, flatten=False):
    """Yield normalized (xb, yb) batches straight from GPU memory."""
    n = X.shape[0]
    if shuffle:
        order = torch.randperm(n, device=X.device, generator=generator)
    else:
        order = torch.arange(n, device=X.device)
    for i in range(0, n, batch_size):
        idx = order[i:i + batch_size]
        xb  = normalize(X[idx])
        if flatten:
            xb = xb.reshape(xb.shape[0], -1)
        yield xb, Y[idx]


---
# Task 1: Backpropagation

**(a)** `Linear`, `ReLU` and softmax cross-entropy with explicit forward/backward, no autograd.
**(b)** Verify the gradients against autograd and against centred finite differences.
**(c)** Train on flattened CIFAR-10 with SGD.
**(d)** Per-layer gradient norms at initialisation, sigmoid vs ReLU.

### (a) Layers with explicit forward/backward

In [ ]:
class Linear:
    """y = x @ W + b, with hand-written backward."""

    def __init__(self, n_in, n_out, init='kaiming', dtype=torch.float32,
                 device=DEVICE, generator=None):
        if init == 'kaiming':     # for ReLU: Var[W] = 2 / fan_in
            std = math.sqrt(2.0 / n_in)
        elif init == 'xavier':    # for sigmoid/tanh: Var[W] = 1 / fan_in
            std = math.sqrt(1.0 / n_in)
        else:
            raise ValueError(init)

        self.W = torch.randn(n_in, n_out, generator=generator, dtype=dtype, device=device) * std
        self.b = torch.zeros(n_out, dtype=dtype, device=device)
        self.dW = torch.zeros_like(self.W)
        self.db = torch.zeros_like(self.b)
        self.x  = None

    def forward(self, x):
        self.x = x                      # needed for dL/dW
        return x @ self.W + self.b

    def backward(self, g):
        self.dW = self.x.T @ g          # (in, N) @ (N, out) -> (in, out)
        self.db = g.sum(dim=0)          # bias is broadcast over the batch
        return g @ self.W.T             # (N, out) @ (out, in) -> (N, in)

    def params(self): return {'W': self.W,  'b': self.b}
    def grads(self):  return {'W': self.dW, 'b': self.db}


class ReLU:
    def forward(self, x):
        self.mask = (x > 0)             # cheaper to store than x itself
        return x * self.mask

    def backward(self, g):
        return g * self.mask

    def params(self): return {}
    def grads(self):  return {}


class Sigmoid:
    """Only needed for Task 1(d)."""
    def forward(self, x):
        self.y = torch.sigmoid(x)       # store the OUTPUT: y' = y(1-y)
        return self.y

    def backward(self, g):
        return g * self.y * (1.0 - self.y)

    def params(self): return {}
    def grads(self):  return {}


class SoftmaxCrossEntropy:
    """Fused softmax + cross-entropy, averaged over the batch."""

    def forward(self, z, y):
        zs       = z - z.max(dim=1, keepdim=True).values      # stability shift
        exp      = zs.exp()
        sum_exp  = exp.sum(dim=1, keepdim=True)
        self.p   = exp / sum_exp                              # softmax probabilities
        self.y   = y
        self.N   = z.shape[0]
        rows     = torch.arange(self.N, device=z.device)
        log_p_correct = zs[rows, y] - sum_exp.squeeze(1).log()
        return -log_p_correct.mean()

    def backward(self):
        g = self.p.clone()
        g[torch.arange(self.N, device=g.device), self.y] -= 1.0
        return g / self.N

In [ ]:
class MLP:
    """sizes=[3072, 512, 256, 10] -> Linear/act/Linear/act/Linear."""

    def __init__(self, sizes, act='relu', init=None, dtype=torch.float32,
                 device=DEVICE, generator=None):
        if init is None:
            init = 'kaiming' if act == 'relu' else 'xavier'
        Act = {'relu': ReLU, 'sigmoid': Sigmoid}[act]

        self.layers = []
        for i in range(len(sizes) - 1):
            self.layers.append(Linear(sizes[i], sizes[i + 1], init=init, dtype=dtype,
                                      device=device, generator=generator))
            if i < len(sizes) - 2:                 # no activation after the output layer
                self.layers.append(Act())

    def forward(self, x):
        for layer in self.layers:
            x = layer.forward(x)
        return x

    def backward(self, g):
        for layer in reversed(self.layers):        # the whole point: reverse order
            g = layer.backward(g)
        return g

    def linears(self):
        return [l for l in self.layers if isinstance(l, Linear)]

    def param_list(self):
        """[(layer, name)] pairs, for the optimizer and the gradient checker."""
        out = []
        for l in self.linears():
            out.append((l, 'W'))
            out.append((l, 'b'))
        return out

    def n_params(self):
        return sum(l.W.numel() + l.b.numel() for l in self.linears())


class SGDManual:
    """Plain SGD written out, so Task 1 uses no torch.optim either."""

    def __init__(self, model, lr):
        self.model, self.lr = model, lr

    @torch.no_grad()
    def step(self):
        for layer, name in self.model.param_list():
            layer.params()[name] -= self.lr * layer.grads()[name]


gen  = torch.Generator(device=DEVICE).manual_seed(SEED)
demo = MLP([3072, 512, 256, 10], act='relu', generator=gen)
print('MLP 3072-512-256-10 has %s parameters' % format(demo.n_params(), ','))

### (b) Gradient verification

In [ ]:
# ---- Check 1: against autograd ------------------------------------------------------

def check_against_autograd(sizes=(256, 64, 32, 10), N=16, seed=0):
    gen   = torch.Generator(device=DEVICE).manual_seed(seed)
    model = MLP(list(sizes), act='relu', dtype=torch.float64, device=DEVICE, generator=gen)
    crit  = SoftmaxCrossEntropy()

    x = torch.randn(N, sizes[0], dtype=torch.float64, device=DEVICE, generator=gen)
    y = torch.randint(0, sizes[-1], (N,), device=DEVICE, generator=gen)

    # --- mine
    loss_mine = crit.forward(model.forward(x), y)
    model.backward(crit.backward())

    # --- PyTorch's, same weights
    mods = []
    for i, l in enumerate(model.linears()):
        m = nn.Linear(l.W.shape[0], l.W.shape[1]).to(DEVICE).double()
        m.weight.data.copy_(l.W.T)      # nn.Linear is (out, in); mine is (in, out)
        m.bias.data.copy_(l.b)
        mods.append(m)
        if i < len(model.linears()) - 1:
            mods.append(nn.ReLU())
    ref = nn.Sequential(*mods)

    loss_ref = F.cross_entropy(ref(x), y)
    loss_ref.backward()

    print('loss  mine = %.12f   autograd = %.12f   allclose = %s'
          % (loss_mine.item(), loss_ref.item(), torch.allclose(loss_mine, loss_ref)))
    print()
    print('%-8s %-14s %-14s %s' % ('layer', 'max|dW diff|', 'max|db diff|', 'allclose(W, b)'))
    all_ok = True
    for i, l in enumerate(model.linears()):
        dW_ref = mods[2 * i].weight.grad.T
        db_ref = mods[2 * i].bias.grad
        okW = torch.allclose(l.dW, dW_ref, rtol=1e-10, atol=1e-12)
        okb = torch.allclose(l.db, db_ref, rtol=1e-10, atol=1e-12)
        all_ok = all_ok and okW and okb
        print('%-8d %-14.3e %-14.3e %s, %s'
              % (i + 1, (l.dW - dW_ref).abs().max().item(),
                 (l.db - db_ref).abs().max().item(), okW, okb))
    print()
    print('ALL GRADIENTS MATCH AUTOGRAD:', all_ok)
    return all_ok

_ = check_against_autograd()

In [ ]:
# ---- Check 2: centred finite differences --------------------------------------------

def check_finite_differences(sizes=(256, 64, 32, 10), N=8, eps=1e-5,
                             n_per_tensor=12, seed=1, verbose=True):
    gen   = torch.Generator(device=DEVICE).manual_seed(seed)
    model = MLP(list(sizes), act='relu', dtype=torch.float64, device=DEVICE, generator=gen)
    crit  = SoftmaxCrossEntropy()

    x = torch.randn(N, sizes[0], dtype=torch.float64, device=DEVICE, generator=gen)
    y = torch.randint(0, sizes[-1], (N,), device=DEVICE, generator=gen)

    def loss_now():
        return crit.forward(model.forward(x), y).item()

    # analytic gradients once, then cloned so later forwards cannot disturb them
    crit.forward(model.forward(x), y)
    model.backward(crit.backward())
    analytic = {(i, nm): model.linears()[i].grads()[nm].clone()
                for i in range(len(model.linears())) for nm in ('W', 'b')}

    rows = []
    rng  = np.random.default_rng(seed)
    for i, layer in enumerate(model.linears()):
        for nm in ('W', 'b'):
            P = layer.params()[nm].view(-1)
            G = analytic[(i, nm)].view(-1)
            errs = []
            for k in rng.choice(P.numel(), size=min(n_per_tensor, P.numel()), replace=False):
                k = int(k)
                orig = P[k].item()
                P[k] = orig + eps; lp = loss_now()
                P[k] = orig - eps; lm = loss_now()
                P[k] = orig                                   # restore exactly
                numeric = (lp - lm) / (2 * eps)
                ana     = G[k].item()
                errs.append(abs(ana - numeric) / (abs(ana) + abs(numeric) + 1e-12))
            rows.append(('layer %d  %s' % (i + 1, nm), float(np.max(errs)), float(np.median(errs))))

    if verbose:
        print('centred finite differences, float64, eps = %g, %d params per tensor'
              % (eps, n_per_tensor))
        print()
        print('%-16s %-14s %-14s %s' % ('tensor', 'max rel err', 'median rel err', 'pass (<1e-7)'))
        for name, mx, med in rows:
            print('%-16s %-14.3e %-14.3e %s' % (name, mx, med, mx < 1e-7))
        print()
        print('WORST relative error overall: %.3e' % max(r[1] for r in rows))
    return rows

fd_rows = check_finite_differences()
save_result('t1b_finite_diff', {'rows': fd_rows})

### (c) Training the MLP on flattened CIFAR-10

In [ ]:
@torch.no_grad()
def evaluate_mlp(model, X, Y, batch_size=1000):
    correct = 0
    for xb, yb in iterate(X, Y, batch_size, shuffle=False, flatten=True):
        correct += (model.forward(xb).argmax(dim=1) == yb).sum().item()
    return correct / X.shape[0]


def train_task1c(sizes=(3072, 512, 256, 10), epochs=30, lr=0.1, batch_size=128):
    gen   = torch.Generator(device=DEVICE).manual_seed(SEED)
    model = MLP(list(sizes), act='relu', generator=gen)
    crit  = SoftmaxCrossEntropy()
    opt   = SGDManual(model, lr=lr)

    hist = {'epoch': [], 'train_loss': [], 'train_acc': [], 'val_acc': [],
            'lr': lr, 'sizes': list(sizes), 'batch_size': batch_size,
            'n_params': model.n_params()}

    for ep in range(1, epochs + 1):
        running, nb, correct, seen = 0.0, 0, 0, 0
        for xb, yb in iterate(X_tr, Y_tr, batch_size, shuffle=True,
                              generator=gen, flatten=True):
            logits = model.forward(xb)
            loss   = crit.forward(logits, yb)
            model.backward(crit.backward())        # <- hand-written backprop
            opt.step()

            running += loss.item(); nb += 1
            correct += (logits.argmax(dim=1) == yb).sum().item(); seen += yb.numel()

        va = evaluate_mlp(model, X_va, Y_va)
        hist['epoch'].append(ep)
        hist['train_loss'].append(running / nb)
        hist['train_acc'].append(correct / seen)
        hist['val_acc'].append(va)
        if ep == 1 or ep % 5 == 0:
            print('epoch %2d  train loss %.4f  train acc %.4f  val acc %.4f'
                  % (ep, running / nb, correct / seen, va))

    hist['final_val_acc'] = hist['val_acc'][-1]
    hist['best_val_acc']  = max(hist['val_acc'])
    return hist

t1c = cached('t1c_mlp_sgd', train_task1c)
print()
print('parameters      : %s' % format(t1c['n_params'], ','))
print('final val acc   : %.4f' % t1c['final_val_acc'])
print('best  val acc   : %.4f' % t1c['best_val_acc'])
print('train/val gap   : %.4f' % (t1c['train_acc'][-1] - t1c['final_val_acc']))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].plot(t1c['epoch'], t1c['train_loss'], color='#2563eb')
ax[0].set_xlabel('epoch'); ax[0].set_ylabel('training loss')
ax[0].set_title('Task 1(c): training loss, hand-written backprop + SGD')
ax[0].grid(alpha=.3)

ax[1].plot(t1c['epoch'], [a * 100 for a in t1c['train_acc']], label='train', color='#2563eb')
ax[1].plot(t1c['epoch'], [a * 100 for a in t1c['val_acc']],   label='validation', color='#dc2626')
ax[1].set_xlabel('epoch'); ax[1].set_ylabel('accuracy (%)')
ax[1].set_title('Accuracy (final val = %.2f%%)' % (100 * t1c['final_val_acc']))
ax[1].legend(); ax[1].grid(alpha=.3)

plt.tight_layout(); plt.show()

### (d) Per-layer gradient norms at initialisation

In [ ]:
def layer_grad_norms(act, depth=10, width=128, N=256, seed=0):
    gen   = torch.Generator(device=DEVICE).manual_seed(seed)
    sizes = [3072] + [width] * (depth - 1) + [10]
    model = MLP(sizes, act=act, generator=gen)       # init defaults to the matching one
    crit  = SoftmaxCrossEntropy()

    xb = normalize(X_tr[:N]).reshape(N, -1)
    yb = Y_tr[:N]

    crit.forward(model.forward(xb), yb)
    model.backward(crit.backward())                  # one backward pass, no update
    return [l.dW.norm().item() for l in model.linears()]


def run_task1d():
    return {'sigmoid': layer_grad_norms('sigmoid'),
            'relu':    layer_grad_norms('relu'),
            'depth': 10, 'width': 128}

t1d = cached('t1d_grad_norms', run_task1d)
print()
print('%-7s %-16s %-16s' % ('layer', 'sigmoid', 'ReLU'))
for i, (s, r) in enumerate(zip(t1d['sigmoid'], t1d['relu']), start=1):
    print('%-7d %-16.3e %-16.3e' % (i, s, r))

print()
print('last layer / first layer  --  sigmoid: %.3e    ReLU: %.3e'
      % (t1d['sigmoid'][-1] / max(t1d['sigmoid'][0], 1e-300),
         t1d['relu'][-1]    / max(t1d['relu'][0],    1e-300)))

In [ ]:
layers = list(range(1, t1d['depth'] + 1))
plt.figure(figsize=(7.5, 5))
plt.semilogy(layers, t1d['sigmoid'], 'o-', color='#dc2626', label='sigmoid + Xavier init')
plt.semilogy(layers, t1d['relu'],    's-', color='#2563eb', label='ReLU + Kaiming init')
plt.xlabel('layer index  (1 = input side, 10 = output side)')
plt.ylabel(r'$\|\partial L / \partial W^{(l)}\|_2$   (log scale)')
plt.title('Task 1(d): per-layer gradient norm at initialization, 10-layer MLP')
plt.xticks(layers); plt.grid(alpha=.3, which='both'); plt.legend()
plt.tight_layout(); plt.show()

---
# Task 2: Optimizers

**(a)** `MySGD` (momentum, weight decay) and `MyAdam` (bias correction) as
`torch.optim.Optimizer` subclasses, matched against `torch.optim`.
**(b)** Trajectories on $f(x,y)=\tfrac12(x^2+50y^2)$.
**(c)** A small CNN with each optimizer at three learning rates.

### (a) The two optimizers

In [ ]:
class MySGD(torch.optim.Optimizer):
    """SGD with optional momentum and weight decay, matching torch.optim.SGD."""

    def __init__(self, params, lr, momentum=0.0, weight_decay=0.0):
        if lr <= 0.0:
            raise ValueError('lr must be positive')
        super().__init__(params, dict(lr=lr, momentum=momentum, weight_decay=weight_decay))

    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            lr, mu, wd = group['lr'], group['momentum'], group['weight_decay']
            for p in group['params']:
                if p.grad is None:
                    continue
                g = p.grad
                if wd != 0.0:
                    g = g.add(p, alpha=wd)           # L2 penalty folded into the gradient
                if mu != 0.0:
                    st = self.state[p]
                    if 'buf' not in st:
                        st['buf'] = g.clone()        # first step: the buffer IS the gradient
                    else:
                        st['buf'].mul_(mu).add_(g)   # b <- mu*b + g
                    g = st['buf']
                p.add_(g, alpha=-lr)


class MyAdam(torch.optim.Optimizer):
    """Adam with bias correction, matching torch.optim.Adam."""

    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0):
        super().__init__(params, dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay))

    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            lr, (b1, b2) = group['lr'], group['betas']
            eps, wd = group['eps'], group['weight_decay']
            for p in group['params']:
                if p.grad is None:
                    continue
                g = p.grad
                if wd != 0.0:
                    g = g.add(p, alpha=wd)
                st = self.state[p]
                if len(st) == 0:                      # lazy state init
                    st['t'] = 0
                    st['m'] = torch.zeros_like(p)
                    st['v'] = torch.zeros_like(p)
                st['t'] += 1
                t, m, v = st['t'], st['m'], st['v']

                m.mul_(b1).add_(g, alpha=1 - b1)          # first moment  (mean)
                v.mul_(b2).addcmul_(g, g, value=1 - b2)   # second moment (uncentred variance)

                bc1 = 1 - b1 ** t                          # bias corrections
                bc2 = 1 - b2 ** t
                denom = (v / bc2).sqrt_().add_(eps)
                p.addcdiv_(m / bc1, denom, value=-lr)

### Matching `torch.optim`

In [ ]:
def _tiny_model(seed):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(20, 32), nn.Tanh(), nn.Linear(32, 5)).double().to(DEVICE)


def compare_optimizers(make_mine, make_ref, name, steps=50, seed=0):
    m1, m2 = _tiny_model(seed), _tiny_model(seed)
    o1, o2 = make_mine(m1.parameters()), make_ref(m2.parameters())

    gen, worst = torch.Generator(device=DEVICE).manual_seed(123), 0.0
    for _ in range(steps):
        x = torch.randn(16, 20, dtype=torch.float64, device=DEVICE, generator=gen)
        y = torch.randint(0, 5, (16,), device=DEVICE, generator=gen)
        for mdl, opt in ((m1, o1), (m2, o2)):
            opt.zero_grad()
            F.cross_entropy(mdl(x), y).backward()
            opt.step()
        for p1, p2 in zip(m1.parameters(), m2.parameters()):
            worst = max(worst, (p1 - p2).abs().max().item())

    ok = all(torch.allclose(p1, p2, rtol=1e-12, atol=1e-13)
             for p1, p2 in zip(m1.parameters(), m2.parameters()))
    print('%-26s max|diff| over %d steps = %.3e   allclose = %s' % (name, steps, worst, ok))
    return ok

checks = [
    compare_optimizers(lambda p: MySGD(p, lr=0.1),
                       lambda p: torch.optim.SGD(p, lr=0.1), 'SGD (plain)'),
    compare_optimizers(lambda p: MySGD(p, lr=0.1, momentum=0.9),
                       lambda p: torch.optim.SGD(p, lr=0.1, momentum=0.9), 'SGD + momentum 0.9'),
    compare_optimizers(lambda p: MySGD(p, lr=0.1, momentum=0.9, weight_decay=5e-4),
                       lambda p: torch.optim.SGD(p, lr=0.1, momentum=0.9, weight_decay=5e-4),
                       'SGD + momentum + wd'),
    compare_optimizers(lambda p: MyAdam(p, lr=1e-3),
                       lambda p: torch.optim.Adam(p, lr=1e-3), 'Adam'),
    compare_optimizers(lambda p: MyAdam(p, lr=1e-3, betas=(0.8, 0.99)),
                       lambda p: torch.optim.Adam(p, lr=1e-3, betas=(0.8, 0.99)),
                       'Adam betas=(0.8,0.99)'),
    compare_optimizers(lambda p: MyAdam(p, lr=1e-3, weight_decay=1e-2),
                       lambda p: torch.optim.Adam(p, lr=1e-3, weight_decay=1e-2),
                       'Adam + weight decay'),
]
print()
print('ALL SIX CONFIGURATIONS MATCH:', all(checks))

### (b) Trajectories on an ill-conditioned quadratic

In [ ]:
def f_val(x, y):
    return 0.5 * (x ** 2 + 50.0 * y ** 2)


def trajectory(opt_factory, start=(-9.0, 1.0), steps=60):
    """Run an optimizer on f and record the path taken."""
    p = torch.tensor([start[0], start[1]], dtype=torch.float64,
                     device=DEVICE, requires_grad=True)
    opt = opt_factory([p])
    path = [p.detach().cpu().numpy().copy()]
    for _ in range(steps):
        opt.zero_grad()
        f_val(p[0], p[1]).backward()
        opt.step()
        path.append(p.detach().cpu().numpy().copy())
    return np.array(path)


LR_SGD, LR_MOM, LR_ADAM = 0.038, 0.038, 0.3
paths = {
    'SGD  (lr=%.3f)' % LR_SGD:              trajectory(lambda p: MySGD(p, lr=LR_SGD)),
    'SGD + momentum  (lr=%.3f)' % LR_MOM:   trajectory(lambda p: MySGD(p, lr=LR_MOM, momentum=0.9)),
    'Adam  (lr=%.1f)' % LR_ADAM:            trajectory(lambda p: MyAdam(p, lr=LR_ADAM)),
}

print('stability limit for plain SGD on this f:  lr < 2/50 = %.3f' % (2 / 50))
print()
print('%-28s %-13s %-12s %s' % ('optimizer', 'final f', 'final |x|', 'sign flips in y'))
for k, P in paths.items():
    flips = int(np.sum(np.diff(np.sign(P[:, 1])) != 0))
    print('%-28s %-13.4e %-12.4f %d' % (k, f_val(P[-1, 0], P[-1, 1]), abs(P[-1, 0]), flips))

In [ ]:
xs = np.linspace(-10, 10, 400)
ys = np.linspace(-1.4, 1.4, 400)
XX, YY = np.meshgrid(xs, ys)
ZZ = f_val(XX, YY)

fig, axes = plt.subplots(2, 1, figsize=(11, 8.5),
                         gridspec_kw={'height_ratios': [1.25, 1]})

ax = axes[0]
levels = [0.5, 2, 5, 10, 20, 40, 65, 100]
cs = ax.contour(XX, YY, ZZ, levels=levels, colors='#9ca3af', linewidths=.8)
ax.clabel(cs, inline=True, fontsize=7, fmt='%g')
for (label, P), col in zip(paths.items(), ['#dc2626', '#2563eb', '#059669']):
    ax.plot(P[:, 0], P[:, 1], 'o-', color=col, ms=2.6, lw=1.2, label=label, alpha=.9)
ax.plot(0, 0, 'k*', ms=14, label='minimum (0, 0)')
ax.plot(-9, 1, 'ks', ms=7, label='start (-9, 1)')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title(r'Task 2(b): optimizer trajectories on $f(x,y)=\frac{1}{2}(x^2+50y^2)$, 60 steps')
ax.legend(loc='upper right', fontsize=8); ax.grid(alpha=.25)

ax = axes[1]
for (label, P), col in zip(paths.items(), ['#dc2626', '#2563eb', '#059669']):
    ax.semilogy(f_val(P[:, 0], P[:, 1]), color=col, lw=1.6, label=label)
ax.set_xlabel('step'); ax.set_ylabel('f(x, y)   (log scale)')
ax.set_title('Function value vs step')
ax.legend(fontsize=8); ax.grid(alpha=.25, which='both')

plt.tight_layout(); plt.show()

### (c) Learning-rate sweep on a small CNN

In [ ]:
class SmallCNN(nn.Module):
    """Three conv stages with batch norm, global average pooling, one linear head."""

    def __init__(self, num_classes=10, p_drop=0.0):
        super().__init__()
        def block(cin, cout):
            return [nn.Conv2d(cin, cout, 3, padding=1, bias=False),
                    nn.BatchNorm2d(cout), nn.ReLU(inplace=True)]
        self.features = nn.Sequential(
            *block(3, 32),   *block(32, 32),  nn.MaxPool2d(2),    # 32 -> 16
            *block(32, 64),  *block(64, 64),  nn.MaxPool2d(2),    # 16 -> 8
            *block(64, 128), nn.MaxPool2d(2),                     # 8  -> 4
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(p_drop) if p_drop > 0 else nn.Identity()
        self.fc   = nn.Linear(128, num_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.fc(self.drop(x))


@torch.no_grad()
def evaluate_cnn(model, X, Y, batch_size=1000):
    model.eval()
    correct = 0
    for xb, yb in iterate(X, Y, batch_size, shuffle=False):
        correct += (model(xb).argmax(1) == yb).sum().item()
    model.train()
    return correct / X.shape[0]


def train_cnn(opt_name, lr, epochs=15, batch_size=128, seed=SEED):
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    model = SmallCNN().to(DEVICE)
    gen   = torch.Generator(device=DEVICE).manual_seed(seed)

    if   opt_name == 'SGD':          opt = MySGD(model.parameters(), lr=lr)
    elif opt_name == 'SGD+momentum': opt = MySGD(model.parameters(), lr=lr, momentum=0.9)
    elif opt_name == 'Adam':         opt = MyAdam(model.parameters(), lr=lr)
    else: raise ValueError(opt_name)

    hist = {'train_loss': [], 'val_acc': [], 'opt': opt_name, 'lr': lr, 'epochs': epochs}
    for ep in range(1, epochs + 1):
        run, nbat = 0.0, 0
        for xb, yb in iterate(X_tr, Y_tr, batch_size, shuffle=True, generator=gen):
            opt.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(xb), yb)
            loss.backward()
            opt.step()
            run += loss.item(); nbat += 1
        hist['train_loss'].append(run / nbat)
        hist['val_acc'].append(evaluate_cnn(model, X_va, Y_va))
    hist['best_val_acc']  = max(hist['val_acc'])
    hist['final_val_acc'] = hist['val_acc'][-1]
    return hist


print('SmallCNN parameters:', format(sum(p.numel() for p in SmallCNN().parameters()), ','))

In [ ]:
def run_sweep():
    LRS  = [1e-3, 1e-2, 1e-1]
    OPTS = ['SGD', 'SGD+momentum', 'Adam']
    out = {'lrs': LRS, 'opts': OPTS, 'runs': {}}
    for o in OPTS:
        for lr in LRS:
            h = train_cnn(o, lr)
            out['runs']['%s@%g' % (o, lr)] = h
            print('%-14s lr=%-7g best val %.4f  final train loss %.4f'
                  % (o, lr, h['best_val_acc'], h['train_loss'][-1]), flush=True)
    return out

t2c = cached('t2c_lr_sweep', run_sweep)

print()
print('best validation accuracy (15 epochs)')
print('%-16s %-10s %-10s %-10s' % ('optimizer', 'lr=0.001', 'lr=0.01', 'lr=0.1'))
for o in t2c['opts']:
    vals = ['%.4f' % t2c['runs']['%s@%g' % (o, lr)]['best_val_acc'] for lr in t2c['lrs']]
    print('%-16s %-10s %-10s %-10s' % (o, *vals))

print()
best_lr = {}
for o in t2c['opts']:
    b = max(t2c['lrs'], key=lambda lr: t2c['runs']['%s@%g' % (o, lr)]['best_val_acc'])
    best_lr[o] = b
    accs = [t2c['runs']['%s@%g' % (o, lr)]['best_val_acc'] for lr in t2c['lrs']]
    print('%-16s best lr %-8g val acc %.4f    spread across the grid %.1f points'
          % (o, b, max(accs), 100 * (max(accs) - min(accs))))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.4))
cols = {'SGD': '#dc2626', 'SGD+momentum': '#2563eb', 'Adam': '#059669'}

for o in t2c['opts']:
    accs = [100 * t2c['runs']['%s@%g' % (o, lr)]['best_val_acc'] for lr in t2c['lrs']]
    ax[0].semilogx(t2c['lrs'], accs, 'o-', color=cols[o], label=o, lw=1.8, ms=7)
ax[0].set_xlabel('learning rate (log scale)')
ax[0].set_ylabel('best validation accuracy (%)')
ax[0].set_title('Task 2(c): validation accuracy vs learning rate')
ax[0].legend(); ax[0].grid(alpha=.3, which='both')

for o in t2c['opts']:
    h = t2c['runs']['%s@%g' % (o, best_lr[o])]
    ax[1].plot(range(1, len(h['train_loss']) + 1), h['train_loss'],
               color=cols[o], lw=1.8, label='%s (lr=%g)' % (o, best_lr[o]))
ax[1].set_xlabel('epoch'); ax[1].set_ylabel('training loss')
ax[1].set_title('Training loss at each optimizer\'s best learning rate')
ax[1].legend(); ax[1].grid(alpha=.3)

plt.tight_layout(); plt.show()

---
# Task 3: Regularization and residual connections

**(a)** Seven configurations on 5,000 training images: baseline, weight decay, dropout,
flip+crop, cutout, mixup, all combined. Cutout and mixup implemented from scratch.
**(b)** Plain and residual networks of depth $6n+2$ at depths 8, 20 and 56.

### (a) Regularization on 5,000 images

In [ ]:
# ---- a fixed 5,000-image subset of the training split --------------------------
g5  = torch.Generator().manual_seed(SEED)
sub = torch.randperm(X_tr.shape[0], generator=g5)[:5000].to(DEVICE)
X5, Y5 = X_tr[sub], Y_tr[sub]
print('Task 3(a) training subset:', tuple(X5.shape))

ARANGE32 = torch.arange(32, device=DEVICE)


def aug_flip_crop(xb_u8, gen):
    """Random horizontal flip, then a random 32x32 crop from a 4-pixel-padded image.

    Done batched on the GPU: every image gets its own flip decision and crop offset,
    gathered in one advanced-indexing operation rather than a Python loop.
    """
    B, C = xb_u8.shape[0], xb_u8.shape[1]
    flip  = torch.rand(B, device=DEVICE, generator=gen) < 0.5
    xb_u8 = torch.where(flip[:, None, None, None], xb_u8.flip(-1), xb_u8)

    padded = F.pad(xb_u8, (4, 4, 4, 4))                      # B,C,40,40
    oy = torch.randint(0, 9, (B,), device=DEVICE, generator=gen)
    ox = torch.randint(0, 9, (B,), device=DEVICE, generator=gen)
    rows = (oy[:, None] + ARANGE32)[:, None, :, None]        # B,1,32,1
    cols = (ox[:, None] + ARANGE32)[:, None, None, :]        # B,1,1,32
    bidx = torch.arange(B, device=DEVICE)[:, None, None, None]
    cidx = torch.arange(C, device=DEVICE)[None, :, None, None]
    return padded[bidx, cidx, rows, cols]                    # B,C,32,32


def apply_cutout(x, length, gen):
    """Zero one random LxL square per image.

    Applied AFTER normalization, so the hole sits at the per-channel mean rather than
    at black. Squares near a border are clipped, which is the standard behaviour.
    """
    B    = x.shape[0]
    cy   = torch.randint(0, 32, (B,), device=DEVICE, generator=gen)
    cx   = torch.randint(0, 32, (B,), device=DEVICE, generator=gen)
    half = length // 2
    my   = (ARANGE32[None, :] >= (cy[:, None] - half)) & (ARANGE32[None, :] < (cy[:, None] + half))
    mx   = (ARANGE32[None, :] >= (cx[:, None] - half)) & (ARANGE32[None, :] < (cx[:, None] + half))
    hole = my[:, :, None] & mx[:, None, :]                   # B,32,32
    return x * (~hole)[:, None, :, :]


def mixup_batch(x, y, alpha, gen, rng):
    """Blend the batch with a shuffled copy of itself. Returns both label sets and lambda.

    lambda comes from an explicit numpy Generator rather than
    torch.distributions.Beta(...).sample(), which would silently draw from the global
    RNG and ignore the generator passed in -- making the run reproducible only by
    accident.
    """
    lam  = float(rng.beta(alpha, alpha))
    perm = torch.randperm(x.shape[0], device=DEVICE, generator=gen)
    return lam * x + (1 - lam) * x[perm], y, y[perm], lam

In [ ]:
CONFIGS = {
    'baseline':     dict(),
    'weight decay': dict(wd=5e-4),
    'dropout':      dict(drop=0.3),
    'flip+crop':    dict(flipcrop=True),
    'cutout':       dict(cutout=8),
    'mixup':        dict(mixup=1.0),
    'all combined': dict(wd=5e-4, drop=0.3, flipcrop=True, cutout=8, mixup=1.0),
}


def train_reg(name, wd=0.0, drop=0.0, flipcrop=False, cutout=0, mixup=0.0,
              epochs=80, bs=128, lr=0.05, seed=SEED):
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    model = SmallCNN(p_drop=drop).to(DEVICE)
    opt   = MySGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=wd)
    gen   = torch.Generator(device=DEVICE).manual_seed(seed)
    rng   = np.random.default_rng(seed)        # drives mixup's lambda, reproducibly

    h, n = {'name': name, 'train_acc': [], 'val_acc': [], 'train_loss': []}, X5.shape[0]
    for ep in range(epochs):
        model.train()
        order = torch.randperm(n, device=DEVICE, generator=gen)
        run, nb_ = 0.0, 0
        for i in range(0, n, bs):
            idx   = order[i:i + bs]
            xb_u8 = aug_flip_crop(X5[idx], gen) if flipcrop else X5[idx]
            xb    = normalize(xb_u8)
            if cutout:
                xb = apply_cutout(xb, cutout, gen)
            yb = Y5[idx]

            opt.zero_grad(set_to_none=True)
            if mixup > 0:
                xb, ya, yh, lam = mixup_batch(xb, yb, mixup, gen, rng)
                out  = model(xb)
                loss = lam * F.cross_entropy(out, ya) + (1 - lam) * F.cross_entropy(out, yh)
            else:
                loss = F.cross_entropy(model(xb), yb)
            loss.backward(); opt.step()
            run += loss.item(); nb_ += 1

        # train accuracy on CLEAN images, so every config measures the same thing
        h['train_loss'].append(run / nb_)
        h['train_acc'].append(evaluate_cnn(model, X5, Y5))
        h['val_acc'].append(evaluate_cnn(model, X_va, Y_va))

    h['final_train_acc'] = h['train_acc'][-1]
    h['final_val_acc']   = h['val_acc'][-1]
    h['best_val_acc']    = max(h['val_acc'])
    h['gap']             = h['final_train_acc'] - h['final_val_acc']
    return h


def run_t3a():
    out = {'order': list(CONFIGS), 'runs': {}}
    for name, kw in CONFIGS.items():
        h = train_reg(name, **kw)
        out['runs'][name] = h
        print('%-14s train %.4f  val %.4f  gap %+.4f  best val %.4f'
              % (name, h['final_train_acc'], h['final_val_acc'], h['gap'],
                 h['best_val_acc']), flush=True)
    return out

t3a = cached('t3a_regularization', run_t3a)

print()
print('%-15s %-12s %-12s %-10s %s' % ('config', 'train acc', 'val acc', 'gap', 'best val'))
print('-' * 62)
for k in t3a['order']:
    h = t3a['runs'][k]
    print('%-15s %-12.4f %-12.4f %-10.4f %.4f'
          % (k, h['final_train_acc'], h['final_val_acc'], h['gap'], h['best_val_acc']))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
cmap = plt.get_cmap('tab10')
cols = {k: cmap(i) for i, k in enumerate(t3a['order'])}

for k in t3a['order']:
    h = t3a['runs'][k]
    ep = range(1, len(h['val_acc']) + 1)
    ax[0].plot(ep, [100 * a for a in h['train_acc']], color=cols[k], lw=1.4, label=k)
    ax[1].plot(ep, [100 * a for a in h['val_acc']],   color=cols[k], lw=1.4, label=k)
    ax[2].plot(ep, [100 * (t - v) for t, v in zip(h['train_acc'], h['val_acc'])],
               color=cols[k], lw=1.4, label=k)

ax[0].set_title('Training accuracy (clean images)')
ax[1].set_title('Validation accuracy')
ax[2].set_title('Generalization gap (train - val)')
for a in ax:
    a.set_xlabel('epoch'); a.set_ylabel('%'); a.grid(alpha=.3)
ax[1].legend(fontsize=7.5, loc='lower right')
plt.tight_layout(); plt.show()

### (b) Plain vs residual networks

Shortcuts use Option A (stride subsample + zero-padded channels), which has no parameters, so
the two families have identical parameter counts at every depth.

In [ ]:
class CifarBlock(nn.Module):
    """Two 3x3 convolutions. Identical in both families except for the `residual` flag."""

    def __init__(self, cin, cout, stride, residual):
        super().__init__()
        self.residual = residual
        self.c1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False)
        self.b1 = nn.BatchNorm2d(cout)
        self.c2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False)
        self.b2 = nn.BatchNorm2d(cout)
        self.downsample = (stride != 1 or cin != cout)
        self.cin, self.cout, self.stride = cin, cout, stride

    def shortcut(self, x):
        """Option A: subsample and zero-pad channels. No parameters, so the plain and
        residual families have identical parameter counts."""
        if self.downsample:
            x = x[:, :, ::self.stride, ::self.stride]
            pad = self.cout - self.cin
            x = F.pad(x, (0, 0, 0, 0, pad // 2, pad - pad // 2))
        return x

    def forward(self, x):
        out = F.relu(self.b1(self.c1(x)))
        out = self.b2(self.c2(out))
        if self.residual:                 # <-- the ONLY difference between the families
            out = out + self.shortcut(x)
        return F.relu(out)


class CifarNet(nn.Module):
    """Depth 6n+2: stem conv, three stages of n blocks at 16/32/64 channels, GAP, linear."""

    def __init__(self, n, residual, num_classes=10):
        super().__init__()
        self.stem = nn.Conv2d(3, 16, 3, 1, 1, bias=False)
        self.bn   = nn.BatchNorm2d(16)
        blocks, cin = [], 16
        for cout, stride in ((16, 1), (32, 2), (64, 2)):
            for j in range(n):
                blocks.append(CifarBlock(cin, cout, stride if j == 0 else 1, residual))
                cin = cout
        self.blocks = nn.Sequential(*blocks)
        self.fc = nn.Linear(64, num_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = F.relu(self.bn(self.stem(x)))
        x = self.blocks(x)
        return self.fc(F.adaptive_avg_pool2d(x, 1).flatten(1))


for n, d in ((1, 8), (3, 20), (9, 56)):
    p_plain = sum(p.numel() for p in CifarNet(n, False).parameters())
    p_res   = sum(p.numel() for p in CifarNet(n, True).parameters())
    print('depth %2d : plain %9s   residual %9s   identical = %s'
          % (d, format(p_plain, ','), format(p_res, ','), p_plain == p_res))

In [ ]:
def train_depth(n, residual, epochs=30, bs=128, lr=0.1, wd=1e-4, seed=SEED):
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    model = CifarNet(n, residual).to(DEVICE)
    opt   = MySGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    gen   = torch.Generator(device=DEVICE).manual_seed(seed)

    h = {'depth': 6 * n + 2, 'residual': residual, 'train_loss': [], 'val_acc': [],
         'n_params': sum(p.numel() for p in model.parameters())}
    for ep in range(epochs):
        model.train()
        run, nb_ = 0.0, 0
        for xb, yb in iterate(X_tr, Y_tr, bs, shuffle=True, generator=gen):
            opt.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(xb), yb)
            loss.backward(); opt.step()
            run += loss.item(); nb_ += 1
        sched.step()
        h['train_loss'].append(run / nb_)
        h['val_acc'].append(evaluate_cnn(model, X_va, Y_va))
    h['final_train_loss'] = h['train_loss'][-1]
    h['final_val_acc']    = h['val_acc'][-1]
    return h


def run_t3b():
    out = {'depths': [8, 20, 56], 'runs': {}}
    for n, depth in ((1, 8), (3, 20), (9, 56)):
        for residual in (False, True):
            h   = train_depth(n, residual)
            key = '%s-%d' % ('residual' if residual else 'plain', depth)
            out['runs'][key] = h
            print('%-14s params %-9s final train loss %.4f  val acc %.4f'
                  % (key, format(h['n_params'], ','), h['final_train_loss'],
                     h['final_val_acc']), flush=True)
    return out

t3b = cached('t3b_depth', run_t3b)

print()
print('%-14s %-11s %-18s %s' % ('network', 'params', 'final train loss', 'val acc'))
print('-' * 58)
for d in t3b['depths']:
    for fam in ('plain', 'residual'):
        h = t3b['runs']['%s-%d' % (fam, d)]
        print('%-14s %-11s %-18.4f %.4f'
              % ('%s-%d' % (fam, d), format(h['n_params'], ','),
                 h['final_train_loss'], h['final_val_acc']))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6), sharey=True)
shades = {8: '#93c5fd', 20: '#3b82f6', 56: '#1e3a8a'}

for i, (fam, title) in enumerate((('plain', 'Plain networks'), ('residual', 'Residual networks'))):
    for d in t3b['depths']:
        h = t3b['runs']['%s-%d' % (fam, d)]
        ax[i].plot(range(1, len(h['train_loss']) + 1), h['train_loss'],
                   color=shades[d], lw=2, label='%s-%d' % (fam, d))
    ax[i].set_title(title); ax[i].set_xlabel('epoch')
    ax[i].legend(); ax[i].grid(alpha=.3)
ax[0].set_ylabel('training loss')
plt.suptitle('Task 3(b): training loss vs epoch -- identical parameter counts within each depth',
             fontsize=11)
plt.tight_layout(); plt.show()

# the degradation question, stated directly
print('final training loss')
print('%-8s %-12s %-12s' % ('depth', 'plain', 'residual'))
for d in t3b['depths']:
    print('%-8d %-12.4f %-12.4f'
          % (d, t3b['runs']['plain-%d' % d]['final_train_loss'],
               t3b['runs']['residual-%d' % d]['final_train_loss']))

---
# Task 4: Best CNN under 500k parameters

At most 500,000 parameters and at most 20 epochs. Tuned on the validation split; the test set is
evaluated once, at the end.

In [ ]:
class BestNet(nn.Module):
    """Residual network of depth 6n+2 with configurable stage widths.

    Reuses CifarBlock from Task 3(b), so `residual=False` gives the matching plain
    network with an identical parameter count.
    """

    def __init__(self, n=3, widths=(21, 42, 84), residual=True, num_classes=10):
        super().__init__()
        c0 = widths[0]
        self.stem = nn.Conv2d(3, c0, 3, 1, 1, bias=False)
        self.bn   = nn.BatchNorm2d(c0)
        blocks, cin = [], c0
        for cout, stride in zip(widths, (1, 2, 2)):
            for j in range(n):
                blocks.append(CifarBlock(cin, cout, stride if j == 0 else 1, residual))
                cin = cout
        self.blocks = nn.Sequential(*blocks)
        self.fc = nn.Linear(widths[-1], num_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = F.relu(self.bn(self.stem(x)))
        x = self.blocks(x)
        return self.fc(F.adaptive_avg_pool2d(x, 1).flatten(1))


PARAM_BUDGET = 500_000
n_res   = sum(p.numel() for p in BestNet(residual=True).parameters())
n_plain = sum(p.numel() for p in BestNet(residual=False).parameters())

assert n_res <= PARAM_BUDGET, 'over the 500k parameter budget'
print('residual depth 20, widths 21/42/84')
print('parameters %s of %s allowed (%.1f%%)   plain counterpart %s, identical = %s'
      % (format(n_res, ','), format(PARAM_BUDGET, ','),
         100 * n_res / PARAM_BUDGET, format(n_plain, ','), n_res == n_plain))

In [ ]:
EPOCHS_T4 = 20     # the assignment's hard limit

def train_t4(tag, residual=True, augment=True, wd=5e-4, optimizer='sgd',
             epochs=EPOCHS_T4, bs=128, seed=SEED, return_model=False):
    """Train one configuration of the Task 4 recipe on the 45k training split."""
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    model = BestNet(residual=residual).to(DEVICE)

    if optimizer == 'sgd':
        opt = MySGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=wd)
    elif optimizer == 'adam':
        opt = MyAdam(model.parameters(), lr=1e-3, weight_decay=wd)
    else:
        raise ValueError(optimizer)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    gen = torch.Generator(device=DEVICE).manual_seed(seed)
    h = {'tag': tag, 'train_loss': [], 'val_acc': [],
         'n_params': sum(p.numel() for p in model.parameters())}

    n = X_tr.shape[0]
    for ep in range(epochs):
        model.train()
        order = torch.randperm(n, device=DEVICE, generator=gen)
        run, nb_ = 0.0, 0
        for i in range(0, n, bs):
            idx   = order[i:i + bs]
            xb_u8 = aug_flip_crop(X_tr[idx], gen) if augment else X_tr[idx]
            xb    = normalize(xb_u8)
            if augment:
                xb = apply_cutout(xb, 8, gen)
            yb = Y_tr[idx]

            opt.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(xb), yb)
            loss.backward(); opt.step()
            run += loss.item(); nb_ += 1
        sched.step()
        h['train_loss'].append(run / nb_)
        h['val_acc'].append(evaluate_cnn(model, X_va, Y_va))

    h['final_val_acc'] = h['val_acc'][-1]
    h['best_val_acc']  = max(h['val_acc'])
    return (h, model) if return_model else h


ABLATIONS = {
    'full recipe':     dict(),
    'minus shortcut':  dict(residual=False),
    'minus augment':   dict(augment=False),
    'minus wt decay':  dict(wd=0.0),
    'Adam instead':    dict(optimizer='adam'),
}

In [ ]:
MODEL_PATH = os.path.join(RESULTS_DIR, 't4_final_model.pt')

def run_t4():
    out = {'order': list(ABLATIONS), 'runs': {}}
    for tag, kw in ABLATIONS.items():
        if tag == 'full recipe':
            h, model = train_t4(tag, return_model=True, **kw)
            torch.save(model.state_dict(), MODEL_PATH)   # kept for the single test evaluation
        else:
            h = train_t4(tag, **kw)
        out['runs'][tag] = h
        print('%-16s best val %.4f   final val %.4f'
              % (tag, h['best_val_acc'], h['final_val_acc']), flush=True)
    return out

t4 = cached('t4_ablation', run_t4)

full = t4['runs']['full recipe']['best_val_acc']
print()
print('%-18s %-12s %-12s %s' % ('configuration', 'best val', 'final val', 'vs full recipe'))
print('-' * 62)
for tag in t4['order']:
    h = t4['runs'][tag]
    delta = 100 * (h['best_val_acc'] - full)
    print('%-18s %-12.4f %-12.4f %s'
          % (tag, h['best_val_acc'], h['final_val_acc'],
             '--' if tag == 'full recipe' else '%+.2f points' % delta))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.4))
cmap = plt.get_cmap('tab10')
cols = {t: cmap(i) for i, t in enumerate(t4['order'])}

for tag in t4['order']:
    h  = t4['runs'][tag]
    ep = range(1, len(h['val_acc']) + 1)
    ax[0].plot(ep, [100 * a for a in h['val_acc']], color=cols[tag], lw=1.7, label=tag)
    ax[1].plot(ep, h['train_loss'], color=cols[tag], lw=1.7, label=tag)

ax[0].set_xlabel('epoch'); ax[0].set_ylabel('validation accuracy (%)')
ax[0].set_title('Task 4 ablation: validation accuracy')
ax[0].legend(fontsize=8, loc='lower right'); ax[0].grid(alpha=.3)
ax[1].set_xlabel('epoch'); ax[1].set_ylabel('training loss')
ax[1].set_title('Training loss')
ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

### The single test evaluation

Evaluated once and cached, so re-running the notebook reads the stored value.

In [ ]:
def evaluate_test_once():
    """Evaluate the final model on the test set. Called at most once; result is cached."""
    model = BestNet(residual=True).to(DEVICE)
    model.load_state_dict(torch.load(MODEL_PATH))
    test_acc = evaluate_cnn(model, X_te, Y_te)
    val_acc  = evaluate_cnn(model, X_va, Y_va)
    return {'test_acc': test_acc, 'val_acc': val_acc,
            'n_params': sum(p.numel() for p in model.parameters()),
            'epochs': EPOCHS_T4}

t4_test = cached('t4_test_eval', evaluate_test_once)

print('FINAL MODEL   %s parameters (limit 500,000),  %d epochs (limit 20)'
      % (format(t4_test['n_params'], ','), t4_test['epochs']))
print('validation %.2f%%    TEST %.2f%%  (evaluated once)'
      % (100 * t4_test['val_acc'], 100 * t4_test['test_acc']))